In [ ]:

from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType, IntegerType,  TimestampType , StringType, StructType, StructField, BooleanType

schema_type = StructType([StructField('Transaction_ID', StringType(), True),
                       StructField('Customer_ID', StringType(), True),
                       StructField('Category', StringType(), True),
                       StructField('Item', StringType(), True),
                       StructField('Price_Per_Unit',DoubleType(), True),
                       StructField('Quantity', DoubleType(), True),
                       StructField('Total_Spent', DoubleType(), True),
                       StructField('Payment_Method', StringType(), True),
                       StructField('Location', StringType(), True),
                       StructField('Transaction_Date', StringType(), True),
                       StructField('Discount_Applied', BooleanType(), True),
                       StructField('updated_time', TimestampType(), True),
                       StructField('_rescued_data', StringType(), True),
                       StructField('source', StringType(), True),
                       StructField('ingested_time',  TimestampType(), True)])
df = spark.read.table('main.bronze.sales')
df = df.select([col(fn.name).cast(fn.dataType).alias(fn.name) for fn in schema_type.fields])
df.printSchema()

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df = df.withColumn(
    "Price_Per_Unit",
    F.when(
        F.col("Price_Per_Unit").isNull()
        & F.col("Quantity").isNotNull()
        & F.col("Total_Spent").isNotNull()
        & (F.col("Quantity") != 0),
        F.col("Total_Spent") / F.col("Quantity")
    ).otherwise(F.col("Price_Per_Unit"))
)

df.show(10)

In [ ]:
observed_prices = (
    df.select("Price_Per_Unit")
    .filter(F.col("Price_Per_Unit").isNotNull())
    .distinct()
    .orderBy("Price_Per_Unit")
)


window_spec = Window.orderBy("Price_Per_Unit")

price_schedule = observed_prices.withColumn(
    "item_number",
    F.row_number().over(window_spec)
)


category_codes = {
    "Beverages": "BEV",
    "Furniture": "FUR",
    "Milk Products": "MILK",
    "Butchers": "BUT",
    "Food": "FOOD",
    "Computers and electric accessories": "CEA",
    "Patisserie": "PAT",
    "Electric household essentials": "EHE"
}

category_map = F.create_map(
    *[
        F.lit(value)
        for pair in category_codes.items()
        for value in pair
    ]
)


df = df.join(
    price_schedule,
    on="Price_Per_Unit",
    how="left"
)

df = df.withColumn(
    "Item",
    F.when(
        F.col("Item").isNull()
        & F.col("Price_Per_Unit").isNotNull()
        & F.col("Category").isin(*category_codes.keys())
        & F.col("item_number").isNotNull(),
        F.concat(
            F.lit("Item_"),
            F.col("item_number").cast("string"),
            F.lit("_"),
            F.element_at(category_map, F.col("Category"))
        )
    ).otherwise(F.col("Item"))
)

df = df.drop("item_number")
df.show(10)

In [ ]:
from delta.tables import DeltaTable
silver_table = "main.silver.sales"


source_df = df.dropDuplicates(["Transaction_ID"])

if spark.catalog.tableExists(silver_table):
    target = DeltaTable.forName(spark, silver_table)

    (
        target.alias("target")
        .merge(
            source_df.alias("source"),
            "target.`Transaction_ID` = source.`Transaction_ID`"
        )
        .whenNotMatchedInsertAll()
        .execute()
    )
else:

    (
        source_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(silver_table)
    )

new_df = spark.read.table('main.silver.sales')

In [ ]:
new_df = spark.read.table('main.silver.sales')
new_df.show(10)